# Stage 5: RAG Evaluation Framework

> **Runtime Prerequisite:** This notebook requires **Stage 4** to have been executed
> in the same Colab/Jupyter session so that the vector store and processed holdings
> data are available.

Evaluates the 13F RAG pipeline against a **Baseline** (vanilla Gemini 3.1 Flash-Lite, no context) using 10 diverse test queries:

| # | Type | Purpose |
|---|------|---------|
| Q1 | Fact-based | Largest position lookup |
| Q2 | Numerical Extraction | Exact share count |
| Q3 | Comparative / Deep Context | Cross-fund sector exposure |
| Q4 | Sector Concentration | Consensus inflow |
| Q5 | CUSIP Lookup | BM25 precision |
| Q6 | Negative / Hallucination | Non-existent entity |
| Q7 | Negative / Out-of-Scope | Wrong time period |
| Q8 | Temporal — Filing Date | Filing date precision |
| Q9 | Temporal — Cross-Period | Insufficient data detection |
| Q10 | Temporal — Specific Fund | Fund period confirmation |

Each query is scored independently by **two judges** on **Faithfulness (1–5)** and a **Grounding Check** against source files.
The final faithfulness score is the **average** of both judges.

**Generator:** Gemini 3.1 Flash-Lite (Google AI Studio) | **Evaluator:** GPT-4o-mini (GitHub Models / OpenAI) | **Judge:** Gemini 2.5 Pro (Google AI Studio / Google DeepMind)

In [1]:
# ── Set Working Directory ─────────────────────────────────────────────────────
# Ensure working directory is the Model/ folder so all relative paths
# (./data/, ./vector_db/, etc.) resolve consistently across notebooks.
import os, pathlib

_cwd = pathlib.Path.cwd()
# If VS Code opened the workspace root, navigate into Model/
if not (_cwd / "data").exists() and (_cwd / "Model" / "data").exists():
    os.chdir(_cwd / "Model")

os.makedirs("data", exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\user\Documents\1fintech\GenAI\Individual Assignment 2\Model


In [2]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma \
             sentence-transformers chromadb rank_bm25 tabulate flashrank openai \
             google-generativeai


You should consider upgrading via the 'C:\Users\user\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
import os
import re
import json
import logging
import textwrap
from dataclasses import dataclass, field
from typing import List, Optional

# Suppress verbose httpx / httpcore request logs (openai client + HuggingFace Hub)
# Must be set before any library imports that trigger HTTP requests.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

from tabulate import tabulate
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers import EnsembleRetriever
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document

# ── Paths ──────────────────────────────────────────────────────────────────────
CHROMA_DIR      = "./vector_db/local_chroma_storage"
FILINGS_BASE    = "./data/13f_filings/sec-edgar-filings"
PROCESSED_JSON  = "./data/processed_holdings.json"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# Generation is handled by Gemini 3.1 Flash-Lite via Google AI Studio (see scoring cell).
# Evaluation is handled by GPT-4o-mini (GitHub Models) + Gemini 2.5 Pro (Google AI Studio).

print("Imports OK.")


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK.


In [4]:
# ── Test set: 10 diverse evaluation queries ───────────────────────────────────
@dataclass
class TestQuery:
    id: int
    query_type: str
    question: str
    # Keywords that MUST appear in a faithful answer (used by keyword fallback scorer)
    required_terms: List[str] = field(default_factory=list)
    # If True, the correct answer is "no information / does not exist"
    expect_no_data: bool = False

TEST_SET = [
    # ── Factual ───────────────────────────────────────────────────────────────
    TestQuery(
        id=1,
        query_type="Fact-based",
        question="What is Berkshire Hathaway's largest position by market value in its Feb 2026 filing?",
        required_terms=["apple", "aapl", "037833100"],
    ),
    TestQuery(
        id=2,
        query_type="Numerical Extraction",
        question="Exactly how many shares of Apple (AAPL, CUSIP 037833100) does Berkshire Hathaway hold in its Q4 2025 filing? Provide the total share count.",
        required_terms=["037833100", "berkshire"],
    ),
    # ── Cross-fund / semantic ─────────────────────────────────────────────────
    TestQuery(
        id=3,
        query_type="Comparative / Deep Context",
        question="Which of the 5 funds (Berkshire Hathaway, Renaissance Technologies, Scion/Burry, Appaloosa, Bridgewater) has the highest technology sector exposure based on the available filings?",
        required_terms=["renaissance", "bridgewater"],
    ),
    TestQuery(
        id=4,
        query_type="Sector Concentration",
        question="Which sector shows the broadest consensus inflow across all available Feb 2026 filings? Cite specific fund names and holdings.",
        required_terms=[],   # narrative — judged by LLM-as-Judge only
    ),
    # ── CUSIP / ticker precision (BM25 track) ─────────────────────────────────
    TestQuery(
        id=5,
        query_type="CUSIP Lookup (BM25 precision)",
        question="Which funds hold CUSIP 88025U109 (10X Genomics) in their Feb 2026 filings, and what is the reported share count for each?",
        required_terms=["88025u109", "bridgewater"],
    ),
    # ── Negative / hallucination ──────────────────────────────────────────────
    TestQuery(
        id=6,
        query_type="Negative / Edge Case (Hallucination Test)",
        question="What is Michael Burry's position in SkyVault Quantum Computing Inc — a company that does not exist and is not in any filing?",
        required_terms=[],
        expect_no_data=True,
    ),
    TestQuery(
        id=7,
        query_type="Negative / Out-of-Scope Period",
        question="What was Michael Burry's position in Tesla (TSLA) in Q3 2024, prior to the Q4 2025 filings? Provide exact share counts.",
        required_terms=[],
        expect_no_data=True,
    ),
    # ── Temporal ─────────────────────────────────────────────────────────────
    TestQuery(
        id=8,
        query_type="Temporal — Filing Date Precision",
        question="Which funds filed their 13F-HR for Q4 2025 in February 2026, and what is the period of report for each?",
        required_terms=["2026-02", "2025-12-31", "berkshire", "renaissance"],
    ),
    TestQuery(
        id=9,
        query_type="Temporal — Cross-Period Comparison",
        question="For any funds where you have both Q3 2025 and Q4 2025 / early 2026 data, which positions were newly opened (NEW) and which were fully closed (CLOSED) between the two periods?",
        required_terms=[],
        expect_no_data=False,  # corpus has 94 funds with BOTH Q3 and Q4 2025 data — cross-period comparison IS possible
    ),
    TestQuery(
        id=10,
        query_type="Temporal — Specific Fund Period",
        question="What filing date appears on the Scion Asset Management (Michael Burry) 13F submission, and does it confirm activity in the February 2026 filing window?",
        required_terms=["scion", "burry", "2026"],
    ),
]

print(f"Test set loaded: {len(TEST_SET)} queries")
for tq in TEST_SET:
    q_display = (tq.question[:72] + "...") if len(tq.question) > 72 else tq.question
    print(f"  Q{tq.id:02d} [{tq.query_type}]: {q_display}")


Test set loaded: 10 queries
  Q01 [Fact-based]: What is Berkshire Hathaway's largest position by market value in its Feb...
  Q02 [Numerical Extraction]: Exactly how many shares of Apple (AAPL, CUSIP 037833100) does Berkshire ...
  Q03 [Comparative / Deep Context]: Which of the 5 funds (Berkshire Hathaway, Renaissance Technologies, Scio...
  Q04 [Sector Concentration]: Which sector shows the broadest consensus inflow across all available Fe...
  Q05 [CUSIP Lookup (BM25 precision)]: Which funds hold CUSIP 88025U109 (10X Genomics) in their Feb 2026 filing...
  Q06 [Negative / Edge Case (Hallucination Test)]: What is Michael Burry's position in SkyVault Quantum Computing Inc — a c...
  Q07 [Negative / Out-of-Scope Period]: What was Michael Burry's position in Tesla (TSLA) in Q3 2024, prior to t...
  Q08 [Temporal — Filing Date Precision]: Which funds filed their 13F-HR for Q4 2025 in February 2026, and what is...
  Q09 [Temporal — Cross-Period Comparison]: For any funds where you have bot

In [5]:

# ── Initialise embeddings, vector store, BM25, and Reranker ───────────────────
# Note: no local LLM is initialised here — generation is handled by
# Generation is handled by Gemini 3.1 Flash-Lite via Google AI Studio (see scoring/generator cell).

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
)
print(f"Chroma store: {vectorstore._collection.count()} vectors")

# ── Rebuild BM25 Retriever from source JSON ───────────────────────────────────
# BM25Retriever is in-memory only; it must be rebuilt on every session.
print("Rebuilding BM25 retriever ...")
with open(PROCESSED_JSON, "r", encoding="utf-8") as f:
    _raw = json.load(f)
_docs = [Document(page_content=e["text"], metadata=e["metadata"]) for e in _raw]

bm25_retriever = BM25Retriever.from_documents(_docs)
bm25_retriever.k = 10
print(f"BM25Retriever: {len(_docs)} docs, k=10")

# ── EnsembleRetriever: 50% BM25 + 50% Vector ─────────────────────────────────
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)
print(f"EnsembleRetriever: 50% BM25 + 50% Vector — ACTIVE")

# ── Cross-Encoder Reranker (Improvement 15: dual narrow/wide) ────────────────
# Narrow (top_n=8) for entity-specific queries; Wide (top_n=20) for broad
# cross-fund synthesis queries that need fund diversity.
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import FlashrankRerank

try:
    compressor_narrow = FlashrankRerank(top_n=8)
    compressor_wide   = FlashrankRerank(top_n=20)
    reranking_retriever = ContextualCompressionRetriever(
        base_compressor=compressor_narrow,
        base_retriever=hybrid_retriever,
    )
    reranking_retriever_wide = ContextualCompressionRetriever(
        base_compressor=compressor_wide,
        base_retriever=hybrid_retriever,
    )
    USE_RERANKER = True
    print("FlashrankRerank reranker: ACTIVE (narrow=8, wide=20)")
except Exception as _exc:
    reranking_retriever = hybrid_retriever
    reranking_retriever_wide = hybrid_retriever
    USE_RERANKER = False
    print(f"[WARN] FlashrankRerank unavailable ({_exc}); using EnsembleRetriever only")

# ── Improvement 12: Entity-scoped retrieval filter ────────────────────────────
# Addresses cross-entity contamination: when a question targets a specific fund,
# the retriever may return plausible-looking chunks from OTHER funds. A 7B model
# fails to discriminate entity identity under this noise pressure, hallucinating
# cross-entity attributions. This post-retrieval filter restricts chunks to only
# those from the fund(s) mentioned in the question.
#
# For broad/comparative queries ("all funds", "which funds"), no filtering is applied.

# Build alias → canonical fund_name mapping from the loaded corpus
_FUND_ALIAS_MAP = {}  # lowercase alias → canonical fund_name from metadata

_all_fund_names = sorted(set(d.metadata.get("fund_name", "") for d in _docs if d.metadata.get("fund_name")))

for _fn in _all_fund_names:
    _fn_lower = _fn.lower()
    _FUND_ALIAS_MAP[_fn_lower] = _fn

    # Extract base name before parenthetical: "Berkshire Hathaway Inc" from "Berkshire Hathaway Inc (Warren Buffett)"
    if "(" in _fn:
        _base = _fn[:_fn.index("(")].strip()
        _FUND_ALIAS_MAP[_base.lower()] = _fn
        # Also add the parenthetical content: "Warren Buffett", "Michael Burry", "Ray Dalio"
        _paren = _fn[_fn.index("(") + 1 : _fn.index(")")].strip()
        if len(_paren) > 3:
            _FUND_ALIAS_MAP[_paren.lower()] = _fn

    # Add first two significant words as alias (e.g. "berkshire hathaway", "bridgewater associates")
    _cleaned = _fn.split("(")[0] if "(" in _fn else _fn
    for _suffix in ["Inc", "LLC", "Corp", "Ltd", "PLC", "AG", "N.A.", "Co"]:
        _cleaned = _cleaned.replace(_suffix, "")
    _words = _cleaned.strip().split()
    if len(_words) >= 2:
        _two_word = " ".join(_words[:2]).lower().strip()
        if len(_two_word) > 5:
            _FUND_ALIAS_MAP[_two_word] = _fn


def _extract_fund_entities(question):
    """Extract fund names mentioned in a question using the corpus alias map.

    Returns a set of canonical fund_name values.
    Empty set means no specific fund was detected (broad query).
    """
    q_lower = question.lower()

    # Broad-query signals: if the question asks about "all funds" or "which funds",
    # don't restrict to a specific entity
    _BROAD_SIGNALS = ["all funds", "which funds", "across all", "all available", "any funds"]
    if any(sig in q_lower for sig in _BROAD_SIGNALS):
        return set()

    matched = set()
    # Sort aliases longest-first to prefer specific matches over substrings
    for alias in sorted(_FUND_ALIAS_MAP.keys(), key=len, reverse=True):
        if len(alias) <= 4:
            continue  # skip very short aliases to avoid false positives
        if alias in q_lower:
            matched.add(_FUND_ALIAS_MAP[alias])

    return matched


def _entity_filter(docs, target_funds):
    """Filter retrieved docs to only those from the specified fund(s).

    If target_funds is empty (broad query), returns all docs unchanged.
    If filtering would remove ALL docs, falls back to unfiltered (entity
    extraction may have been wrong, or the fund isn't in the corpus).
    """
    if not target_funds:
        return docs

    filtered = [d for d in docs if d.metadata.get("fund_name", "") in target_funds]

    if not filtered:
        # Entity was mentioned but no matching chunks were retrieved — keep originals
        # so the model can at least attempt an answer or correctly state "not found"
        return docs

    return filtered


# ── Improvement 13: Temporal metadata filter ──────────────────────────────────
# Prevents temporal bleeding: when a query targets "Feb 2026 filings", the
# retriever should not return chunks from Q1/Q2/Q3 2025 just because they are
# semantically similar. This filter restricts to the correct reporting period.
#
# Fix: Uses period_of_report as primary filter. filing_date is only used as
# fallback when period_of_report is UNKNOWN, preventing late-filed Q3 filings
# from leaking into Q4 results via the filing_date >= threshold branch.

def _temporal_filter(docs, question):
    """Filter retrieved docs to the time period specified in the question."""
    q_lower = question.lower()

    PERIOD_MAP = {
        ("feb 2026", "february 2026", "q4 2025"): ("2025-12-31", "2026-01-01"),
        ("nov 2025", "november 2025", "q3 2025"): ("2025-09-30", "2025-10-01"),
        ("aug 2025", "august 2025", "q2 2025"):   ("2025-06-30", "2025-07-01"),
        ("may 2025", "q1 2025"):                   ("2025-03-31", "2025-04-01"),
    }

    target_period = None
    min_filing = None
    for signals, (period, filing_min) in PERIOD_MAP.items():
        if any(s in q_lower for s in signals):
            target_period = period
            min_filing = filing_min
            break

    # Fallback: "latest", "most recent", "current" → use the newest period in docs
    if target_period is None:
        _LATEST_SIGNALS = ["latest", "most recent", "current", "newest"]
        if any(s in q_lower for s in _LATEST_SIGNALS):
            periods = [d.metadata.get("period_of_report", "") for d in docs
                       if d.metadata.get("period_of_report", "") not in ("", "UNKNOWN")]
            if periods:
                target_period = max(periods)

    if target_period is None:
        return docs  # broad/ambiguous query — no temporal filter

    filtered = [
        d for d in docs
        if d.metadata.get("period_of_report", "") == target_period
        or (d.metadata.get("period_of_report", "") in ("", "UNKNOWN")
            and d.metadata.get("filing_date", "") >= min_filing)
    ]
    return filtered if filtered else docs


# ── Improvement 14: Fund diversity filter for broad queries ───────────────────
# Ensures no single fund dominates retrieved results, so the Map-Reduce chain
# actually sees data from multiple funds for cross-fund synthesis.

def _diversify_by_fund(docs, max_per_fund=2):
    """Ensure no single fund dominates retrieved results for broad queries."""
    fund_counts = {}
    diverse = []
    for d in docs:
        fund = d.metadata.get("fund_name", "")
        fund_counts[fund] = fund_counts.get(fund, 0) + 1
        if fund_counts[fund] <= max_per_fund:
            diverse.append(d)
    return diverse if diverse else docs


# Quick diagnostic: verify alias map covers test-set entities
_test_aliases = ["berkshire hathaway", "michael burry", "scion", "bridgewater", "ray dalio"]
print(f"\nEntity-scoped filter   : ACTIVE ({len(_FUND_ALIAS_MAP)} aliases for {len(_all_fund_names)} funds)")
for _ta in _test_aliases:
    _match = _FUND_ALIAS_MAP.get(_ta, "NOT FOUND")
    print(f"  '{_ta}' → {_match}")

print("Temporal filter        : ACTIVE (period_of_report primary, filing_date fallback)")
print("Fund diversity filter  : ACTIVE (max 2 chunks per fund for broad queries)")
print("\nAll components ready.")


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5421.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chroma store: 95377 vectors
Rebuilding BM25 retriever ...
BM25Retriever: 95377 docs, k=10
EnsembleRetriever: 50% BM25 + 50% Vector — ACTIVE
FlashrankRerank reranker: ACTIVE (narrow=8, wide=20)

Entity-scoped filter   : ACTIVE (207 aliases for 100 funds)
  'berkshire hathaway' → Berkshire Hathaway Inc (Warren Buffett)
  'michael burry' → Scion Asset Management (Michael Burry)
  'scion' → NOT FOUND
  'bridgewater' → NOT FOUND
  'ray dalio' → Bridgewater Associates (Ray Dalio)
Temporal filter        : ACTIVE (period_of_report primary, filing_date fallback)
Fund diversity filter  : ACTIVE (max 2 chunks per fund for broad queries)

All components ready.


In [6]:
# ── RAG prompt templates and chain routing configuration ──────────────────────
#
# Improvement 2 (chain routing) + Improvement 3 (verbatim preservation) +
# Improvement 4 (query-type dispatch):
#
#   - Map-Reduce chain: used ONLY for comparative/synthesis queries that must
#     integrate information across multiple funds (Q03, Q04).
#   - Stuff chain: used for all exact-lookup queries (fact, numerical, CUSIP,
#     temporal) where passing chunks verbatim preserves atomic values that
#     Map-Reduce's summarisation step would otherwise discard.
#   - All prompts carry an explicit VERBATIM instruction so exact share counts,
#     CUSIPs, dates, and fund names are never paraphrased or rounded.
#
# Improvement 7 (prompt reframing): Prompts reframed from negative ("Do NOT")
#   to positive ("Extract directly") to prevent small-model refusal bias.
# Improvement 8 (extraction cue): "The data above contains the answer." primes
#   the model to extract rather than refuse.

# ── Map prompt (comparative / synthesis queries) ──────────────────────────────
RAG_MAP_TEMPLATE = """You are a financial analyst reviewing a single 13F filing data excerpt.
The data below contains the answer. Extract it directly.
Use ONLY the data shown below — no outside knowledge.
If and ONLY if the specific data point is truly absent from every row below,
state: "Information not available in this excerpt."

CRITICAL: Copy ALL numerical values, CUSIPs, share counts, dates, and fund names
VERBATIM from the source text. Do not paraphrase, round, or restate them.

Context:
{context}

Question: {question}

The data above contains the answer. Answer concisely, citing the fund name and copying exact figures as they appear:"""

# ── Reduce prompt (comparative / synthesis queries) ───────────────────────────
RAG_REDUCE_TEMPLATE = """You are a meticulous financial analyst consolidating 13F filing excerpts.
The summaries below contain the answer. Use ONLY these summaries — no outside knowledge.
If data is genuinely missing or contradictory, state: "Insufficient data in filings."

CRITICAL: Preserve all numerical values, CUSIPs, share counts, dates, and fund names
EXACTLY as they appear in the summaries. Do not round, paraphrase, or invent figures.

Summaries:
{summaries}

Provide a final consolidated answer. For every factual claim, cite:
- The exact fund name
- The exact share count or market value
- The exact CUSIP or ticker
- The exact filing date or report period"""

# ── Stuff prompt (single-fact / exact-lookup queries) ─────────────────────────
RAG_STUFF_TEMPLATE = """You are a precise financial analyst. Answer using ONLY the 13F filing data below.
The filing data below contains the answer. Extract it directly.
If and ONLY if the specific data point is truly absent from every row below,
state: "This information is not present in the retrieved filing data."
Use ONLY the data below — no outside knowledge.

CRITICAL: Copy ALL numerical values, share counts, CUSIPs, dates, and fund names
VERBATIM from the context below. Do not paraphrase, round, or restate them in any way.

Filing data:
{context}

Question: {question}

The data above contains the answer. Answer (cite exact fund name, exact share count, exact CUSIP, exact filing date):"""

# ── Query routing map ─────────────────────────────────────────────────────────
# Map-Reduce is suited to cross-fund synthesis (comparative, sector queries).
# Stuff passes all chunks in one prompt — better for exact value extraction.
MAP_REDUCE_TYPES = {
    "Cross-Fund Sector Synthesis",
    "Comparative / Deep Context",
    "Sector Concentration",
}

print("RAG prompt templates defined.")
print(f"Map-Reduce routes (synthesis) : {MAP_REDUCE_TYPES}")
print("All other query types         : Stuff chain (verbatim extraction).")

RAG prompt templates defined.
Map-Reduce routes (synthesis) : {'Comparative / Deep Context', 'Cross-Fund Sector Synthesis', 'Sector Concentration'}
All other query types         : Stuff chain (verbatim extraction).


In [7]:
import os
import re
import json
import time
import pathlib
import textwrap
import logging
from dataclasses import dataclass, field
from typing import List, Optional

# Suppress verbose httpx / httpcore request logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

import google.generativeai as genai
import openai as _openai

# ── Gemini API key (generator + judge) ────────────────────────────────────────
_GEMINI_KEY_FILE = pathlib.Path(__file__).parent / "gemini_api_key.txt" if "__file__" in dir() \
                   else pathlib.Path("gemini_api_key.txt")

GEMINI_API_KEY = ""
if _GEMINI_KEY_FILE.exists():
    _raw_key = _GEMINI_KEY_FILE.read_text(encoding="utf-8").strip()
    if _raw_key and _raw_key != "PASTE_YOUR_GEMINI_API_KEY_HERE":
        GEMINI_API_KEY = _raw_key

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "Gemini API key not found. Either:\n"
        "  1. Paste your key into  Model/gemini_api_key.txt  (one line, no quotes), or\n"
        "  2. Set the environment variable: $env:GEMINI_API_KEY = 'your_key'"
    )

print(f"Gemini API key loaded from: {'gemini_api_key.txt' if _GEMINI_KEY_FILE.exists() and GEMINI_API_KEY else 'environment variable'}")

genai.configure(api_key=GEMINI_API_KEY)

GENERATOR_MODEL_NAME = "gemini-3.1-flash-lite-preview"
_gemini_generator = genai.GenerativeModel(
    model_name=GENERATOR_MODEL_NAME,
    generation_config=genai.GenerationConfig(
        temperature=0.0,
        max_output_tokens=1024,
    ),
)

JUDGE_MODEL_NAME = "gemini-2.5-pro"
_gemini_judge = genai.GenerativeModel(
    model_name=JUDGE_MODEL_NAME,
    generation_config=genai.GenerationConfig(
        temperature=0.0,
        max_output_tokens=65536,
    ),
)

# ── GitHub Models endpoint (evaluator) ────────────────────────────────────────
_GITHUB_TOKEN_FILE = pathlib.Path(__file__).parent / "github_token.txt" if "__file__" in dir() \
                     else pathlib.Path("github_token.txt")

GITHUB_TOKEN = ""
if _GITHUB_TOKEN_FILE.exists():
    _raw_gh = _GITHUB_TOKEN_FILE.read_text(encoding="utf-8").strip()
    if _raw_gh and _raw_gh != "PASTE_YOUR_GITHUB_TOKEN_HERE":
        GITHUB_TOKEN = _raw_gh

if not GITHUB_TOKEN:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

if not GITHUB_TOKEN:
    raise EnvironmentError(
        "GitHub token not found. Either:\n"
        "  1. Paste your token into  Model/github_token.txt  (one line, no quotes), or\n"
        "  2. Set the environment variable: $env:GITHUB_TOKEN = 'your_token'"
    )

print(f"GitHub token loaded from: {'github_token.txt' if _GITHUB_TOKEN_FILE.exists() and GITHUB_TOKEN else 'environment variable'}")

_github_client = _openai.OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=GITHUB_TOKEN,
)

EVALUATOR_MODEL = "gpt-4o-mini"                    # OpenAI — 150 req/day on GitHub Models free tier
JUDGE_MODEL     = JUDGE_MODEL_NAME                  # Gemini 2.5 Pro via Google AI Studio

# ── Rate-limit tracking ──────────────────────────────────────────────────────
_gemini_gen_call_count = 0
_evaluator_call_count = 0
_judge_call_count = 0
_GEMINI_GEN_INTER_CALL_DELAY = 2   # seconds between Gemini generator calls
_EVALUATOR_INTER_CALL_DELAY = 2    # seconds between evaluator calls
_JUDGE_INTER_CALL_DELAY = 2        # seconds between judge calls

# ── Architecture ──────────────────────────────────────────────────────────────
# Generator  : Gemini 3.1 Flash-Lite (Google AI Studio / Google DeepMind)
# Evaluator  : GPT-4o-mini (GitHub Models / OpenAI)
# Judge      : Gemini 2.5 Pro (Google AI Studio / Google DeepMind)
# Independence : Two different providers — no self-grading (different model families).

LLM_JUDGE_TEMPLATE = """You are an expert financial AI evaluator assessing RAG pipeline quality.

Rate the ANSWER below on a 1-5 scale for ACCURACY, SPECIFICITY, and GROUNDING.
Use the PROVIDED CONTEXT as the ground-truth reference for 13F filing data.

CONTEXT (retrieved 13F filing chunks — treat as ground truth):
{context}

QUESTION:
{question}

ANSWER TO EVALUATE:
{answer}

SCORING RUBRIC:
5 = Excellent: answer cites specific data (fund names, share counts, CUSIPs, dates) that are
    directly confirmed by the context. Claims are precise and verifiable.
    Minor formatting differences do not reduce the score.
4 = Good: answer is mostly correct with specific citations. May have one minor inaccuracy
    or omission, but core factual claims are supported by the context.
3 = Adequate: answer is directionally correct and addresses the question, but is missing
    some key specifics that ARE available in the context, or includes partial data.
2 = Poor: answer makes specific claims that contradict the context, OR gives a generic
    response when the context contains specific data that should have been cited.
1 = Fail: answer is entirely wrong, fabricates data not in context, gives an error message,
    OR refuses to answer when the context clearly contains the relevant information.

IMPORTANT for scoring:
- Credit partial answers: if 3 of 4 data points are correct, score 4 not 2.
- A local model may phrase things differently than a cloud model — judge the factual
  content, not the prose style.
- PARTIAL CONTEXT: The context may contain data from DIFFERENT funds or periods than the
  question asks about. If the question asks about Fund X but the context only has Fund Y
  data, an answer that (a) states Fund X data is unavailable AND (b) analyses the Fund Y
  data that IS present should score 3-4. A blanket refusal like 'Insufficient data' with
  no analysis of available data should score 2.
- Fabrication check: an answer that invents specific numbers, dates, or fund names not
  present anywhere in the context should score 1, even if the claims sound plausible.
- An answer that correctly states 'data not available' when the context truly contains
  NO relevant filing data at all should score 4-5 (appropriate refusal).
- Generic/vague answers score lower than specific, data-backed answers.

Respond with ONLY a valid JSON object — no extra text before or after:
{{"score": <integer 1-5>, "rationale": "<one concise sentence>"}}"""

LLM_JUDGE_PROMPT = PromptTemplate(
    template=LLM_JUDGE_TEMPLATE,
    input_variables=["context", "question", "answer"],
)


def _build_judge_prompt(response, tq, context_docs):
    """Build the judge prompt text (shared by both evaluator and judge)."""
    context_text = "\n\n---\n\n".join(
        d.page_content[:1500] for d in context_docs
    ) if context_docs else "(no context retrieved — baseline evaluation)"
    return LLM_JUDGE_PROMPT.format(
        context=context_text,
        question=tq.question,
        answer=response[:1500],
    )


def _call_github_model(model_name, prompt_text, role_label, call_delay):
    """Call a GitHub Models endpoint and parse JSON score response."""
    _MAX_RETRIES = 3
    for _attempt in range(1, _MAX_RETRIES + 1):
        try:
            resp = _github_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": "You are an expert financial AI evaluator. Respond with ONLY valid JSON."},
                    {"role": "user", "content": prompt_text},
                ],
                max_tokens=256,
                temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
            json_match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                score = max(1, min(5, int(parsed.get("score", 3))))
                rationale = str(parsed.get("rationale", "No rationale provided."))
                return score, rationale
            raise ValueError(f"No JSON found in {model_name} response: {raw[:300]}")
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "rate" in err_str.lower() or "quota" in err_str.lower()
            if is_rate_limit and _attempt < _MAX_RETRIES:
                wait = call_delay * _attempt * 3
                print(f"\n  [RATE LIMIT] {role_label} ({model_name}) attempt {_attempt}/{_MAX_RETRIES} "
                      f"hit rate limit. Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            raise  # let caller handle


def _call_gemini_judge(prompt_text, role_label, call_delay):
    """Call Gemini 2.5 Pro as judge via Google AI Studio and parse JSON score response."""
    _MAX_RETRIES = 3
    for _attempt in range(1, _MAX_RETRIES + 1):
        try:
            system = "You are an expert financial AI evaluator. Respond with ONLY valid JSON."
            full_prompt = f"{system}\n\n{prompt_text}"

            gen_config = genai.GenerationConfig(
                temperature=0.0,
                max_output_tokens=65536,
            )
            resp = _gemini_judge.generate_content(
                full_prompt,
                generation_config=gen_config,
            )
            # Handle empty response (thinking tokens can exhaust budget)
            raw = None
            try:
                raw = resp.text.strip()
            except (ValueError, AttributeError):
                # .text throws if no valid Part — try candidates directly
                if resp.candidates and resp.candidates[0].content.parts:
                    raw = resp.candidates[0].content.parts[0].text.strip()
            if not raw:
                _reason = getattr(resp.candidates[0], 'finish_reason', 'unknown') if resp.candidates else 'no_candidates'
                raise ValueError(f"Gemini returned empty response (finish_reason={_reason})")
            json_match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group())
                score = max(1, min(5, int(parsed.get("score", 3))))
                rationale = str(parsed.get("rationale", "No rationale provided."))
                return score, rationale
            raise ValueError(f"No JSON found in {JUDGE_MODEL} response: {raw[:300]}")
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
            is_empty_response = "empty response" in err_str or "finish_reason" in err_str
            if (is_rate_limit or is_empty_response) and _attempt < _MAX_RETRIES:
                wait = call_delay * _attempt * 3
                _label = "RATE LIMIT" if is_rate_limit else "EMPTY RESPONSE"
                print(f"\n  [{_label}] {role_label} ({JUDGE_MODEL}) attempt {_attempt}/{_MAX_RETRIES} "
                      f"— {err_str[:120]}. Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            raise


def score_evaluator(response, tq, context_docs):
    """GPT-4o-mini evaluator via GitHub Models — faithfulness scorer."""
    global _evaluator_call_count

    prompt_text = _build_judge_prompt(response, tq, context_docs)

    if _evaluator_call_count > 0:
        time.sleep(_EVALUATOR_INTER_CALL_DELAY)

    try:
        score, rationale = _call_github_model(
            EVALUATOR_MODEL, prompt_text, "Evaluator", _EVALUATOR_INTER_CALL_DELAY
        )
        _evaluator_call_count += 1
        return score, rationale, "gpt4omini_evaluator"
    except Exception as exc:
        print(f"  [WARN] Evaluator ({EVALUATOR_MODEL}) failed: {str(exc)[:150]}")
        fb_score = _score_faithfulness_keywords(response, tq)
        return fb_score, f"[FALLBACK-KEYWORD] Evaluator error: {exc}", "keyword_fallback"


def score_judge(response, tq, context_docs):
    """Gemini 2.5 Pro judge via Google AI Studio — faithfulness scorer."""
    global _judge_call_count

    prompt_text = _build_judge_prompt(response, tq, context_docs)

    if _judge_call_count > 0:
        time.sleep(_JUDGE_INTER_CALL_DELAY)

    try:
        score, rationale = _call_gemini_judge(
            prompt_text, "Judge", _JUDGE_INTER_CALL_DELAY
        )
        _judge_call_count += 1
        return score, rationale, "gemini2_judge"
    except Exception as exc:
        print(f"  [WARN] Judge ({JUDGE_MODEL}) failed: {str(exc)[:150]}")
        fb_score = _score_faithfulness_keywords(response, tq)
        return fb_score, f"[FALLBACK-KEYWORD] Judge error: {exc}", "keyword_fallback"


def score_dual(response, tq, context_docs):
    """Run both evaluator and judge, return averaged score.
    Returns: (avg_score, eval_score, eval_rat, eval_method,
              judge_score, judge_rat, judge_method)"""
    e_score, e_rat, e_method = score_evaluator(response, tq, context_docs)
    j_score, j_rat, j_method = score_judge(response, tq, context_docs)
    avg = round((e_score + j_score) / 2, 1)
    return avg, e_score, e_rat, e_method, j_score, j_rat, j_method


def _score_faithfulness_keywords(response, tq):
    """Keyword-count fallback scorer (used only when both judges fail)."""
    resp_lower = response.lower()
    if tq.expect_no_data:
        refusal_signals = [
            "not available", "no information", "does not exist",
            "not found", "not present", "cannot find", "no position",
            "not in", "not mentioned", "no data",
        ]
        return 5 if any(s in resp_lower for s in refusal_signals) else 1
    if not tq.required_terms:
        return 3
    hits = [t for t in tq.required_terms if t.lower() in resp_lower]
    ratio = len(hits) / len(tq.required_terms)
    if ratio == 1.0:   return 5
    if ratio >= 0.75:  return 4
    if ratio >= 0.5:   return 3
    if ratio > 0:      return 2
    return 1


# ── Improvement 9: Non-vacuous grounding check ───────────────────────────────
def grounding_check(response, filings_base):
    """Scans source full-submission.txt files for any 9-char CUSIP in the response.
    Returns (bool|None, str): None means grounding is indeterminate (refusal/empty)."""
    resp_lower = response.lower()

    refusal_phrases = [
        "not present", "not available", "no information",
        "does not exist", "not found", "cannot find", "no data",
        "insufficient data", "not mentioned", "no holdings data",
    ]
    is_refusal = any(p in resp_lower for p in refusal_phrases) or not response.strip()

    _all_9char = set(re.findall(r'\b[A-Z0-9]{9}\b', response.upper()))
    cusips_in_response = {t for t in _all_9char if re.search(r'\d', t)}
    if not cusips_in_response:
        if is_refusal:
            return None, "Refusal/empty answer — grounding indeterminate (no CUSIPs to verify)."
        return True, "No specific CUSIPs to verify (narrative answer)."

    corpus = ""
    for root, _, files in os.walk(filings_base):
        for fname in files:
            if fname == "full-submission.txt":
                try:
                    with open(os.path.join(root, fname), "r",
                              encoding="utf-8", errors="ignore") as f:
                        corpus += f.read(5_000_000).upper()
                except OSError:
                    continue

    verified = [c for c in cusips_in_response if c in corpus]
    unverified = [c for c in cusips_in_response if c not in corpus]

    grounded = len(unverified) == 0
    parts = []
    if verified:
        parts.append(f"Verified CUSIPs: {verified}")
    if unverified:
        parts.append(f"UNVERIFIED (possible hallucination): {unverified}")

    return grounded, " | ".join(parts) if parts else "No verifiable identifiers found."

# ── Generator functions (Gemini 3.1 Flash-Lite via Google AI Studio) ──────────

def _call_gemini(system_content, user_content, max_tokens=1024):
    """Call Gemini 3.1 Flash-Lite as the RAG answer generator."""
    global _gemini_gen_call_count

    if _gemini_gen_call_count > 0:
        time.sleep(_GEMINI_GEN_INTER_CALL_DELAY)

    prompt = f"{system_content}\n\n{user_content}"

    for _attempt in range(1, 4):
        try:
            gen_config = genai.GenerationConfig(
                temperature=0.0,
                max_output_tokens=max_tokens,
            )
            resp = _gemini_generator.generate_content(
                prompt,
                generation_config=gen_config,
            )
            _gemini_gen_call_count += 1
            return resp.text.strip()
        except Exception as exc:
            err_str = str(exc)
            is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
            if _attempt < 3:
                wait = 5 * _attempt if not is_rate_limit else 10 * _attempt
                print(f"    [GEMINI GEN RETRY] attempt {_attempt}/3: {err_str[:100]}. "
                      f"Retrying in {wait}s ...")
                time.sleep(wait)
                continue
            return f"[GEMINI ERROR] {exc}"


def run_baseline(question):
    """Baseline: Gemini 3.1 Flash-Lite with NO retrieved context (parametric knowledge only)."""
    system = (
        "You are a financial analyst. Answer the following question about "
        "institutional 13F-HR SEC filings. If you do not have the specific data, "
        "state that clearly rather than guessing."
    )
    return _call_gemini(system, question)


# ── Improvement 11: Pre-filter large chunks to relevant rows ─────────────────
def _prefilter_chunks(question, docs):
    """Keep only table rows matching question keywords; preserve header metadata."""
    keywords = [w.lower() for w in question.split() if len(w) > 3]
    filtered = []
    for d in docs:
        lines = d.page_content.split("\n")
        header_lines = []
        data_lines = []
        for line in lines:
            stripped = line.strip()
            if stripped.startswith("|"):
                if "nameOfIssuer" in line or ":-" in line or "--" in line:
                    header_lines.append(line)
                else:
                    data_lines.append(line)
            else:
                header_lines.append(line)

        matched_rows = [r for r in data_lines if any(k in r.lower() for k in keywords)]

        if matched_rows:
            filtered.append("\n".join(header_lines + matched_rows))
        else:
            filtered.append(d.page_content)
    return filtered


def run_stuff_rag(question, docs):
    """Stuff chain: concatenate all retrieved chunks into one prompt and call Gemini.
    Pre-filters large chunks to relevant rows before prompting (Improvement 11)."""
    filtered_chunks = _prefilter_chunks(question, docs)
    context = "\n\n---\n\n".join(filtered_chunks)
    system = (
        "You are a precise financial analyst. "
        "Use ONLY the 13F filing data provided below — no outside knowledge. "
        "Extract data directly from the tables and metadata headers. "
        "If the specific data point is absent from the provided excerpts, "
        "state: 'This information is not present in the retrieved filing data.' "
        "However, if the data is present for DIFFERENT funds or periods than asked about, "
        "report what IS available and note which requested entities are missing. "
        "CRITICAL: Copy ALL numerical values, share counts, CUSIPs, dates, and fund names "
        "VERBATIM from the context. Do not paraphrase, round, or restate them. "
        "You MUST cite the exact fund name as it appears in the context header (e.g., "
        "'Fund: Treasurer of the State of North Carolina') and the accession number "
        "for every claim. Never use generic labels like 'Fund 1' or 'the fund'."
    )
    user = (
        f"Filing data:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer using ONLY the data above. "
        "Cite exact fund name, accession number, share count, CUSIP, and filing date:"
    )
    return _call_gemini(system, user, max_tokens=1024)


def run_map_reduce_rag(question, docs):
    """Map-Reduce chain with intermediate summary logging (Improvement 16).
    Pre-filters large chunks to relevant rows before map step (Improvement 11).
    Returns (final_answer, map_log) tuple for audit trail."""
    filtered_chunks = _prefilter_chunks(question, docs)
    map_system = (
        "You are a financial analyst reviewing a single 13F filing excerpt. "
        "First, extract the fund name, filing date, period of report, and accession "
        "number from the metadata header. Then extract the holdings data most relevant "
        "to the question — include issuer names, CUSIPs, share counts, market values, "
        "and any sector/industry identifiers present. Copy ALL numerical values and "
        "identifiers VERBATIM. Always extract what IS present — even if the excerpt "
        "is from a different fund than the one asked about. "
        "If the excerpt is completely empty or contains no tabular data, "
        "state: 'No holdings data in this excerpt.'"
    )
    summaries = []
    map_log = []
    for i, chunk_text in enumerate(filtered_chunks):
        map_user = (
            f"Context (excerpt {i+1} of {len(filtered_chunks)}):\n{chunk_text}\n\n"
            f"Question: {question}\n\n"
            "Extract the data described above. Answer concisely, copying exact figures verbatim:"
        )
        summary = _call_gemini(map_system, map_user, max_tokens=400)
        summaries.append(summary)
        fund_name = docs[i].metadata.get("fund_name", "unknown") if i < len(docs) else "unknown"
        chunk_idx = docs[i].metadata.get("chunk_index", "?") if i < len(docs) else "?"
        map_log.append({
            "excerpt": i + 1,
            "fund_name": fund_name,
            "chunk_index": chunk_idx,
            "summary": summary[:500],
        })

    reduce_system = (
        "You are a meticulous financial analyst consolidating multiple 13F filing excerpts. "
        "Use ONLY the summaries below — no outside knowledge. "
        "Preserve all numerical values, CUSIPs, share counts, dates, and fund names "
        "EXACTLY as stated. Do not invent, round, or paraphrase figures. "
        "IMPORTANT: If the question asks about specific funds that are NOT present in "
        "the summaries, state which funds were requested but unavailable, then analyse "
        "the data that IS available. Identify trends: compare values across quarters "
        "if multiple periods are present, note which positions grew or shrank. "
        "Always cite the exact fund name and accession number for every claim."
    )
    joined = "\n\n---\n\n".join(
        f"Excerpt {i+1}:\n{s}" for i, s in enumerate(summaries)
    )
    reduce_user = (
        f"Summaries from {len(filtered_chunks)} retrieved chunks:\n{joined}\n\n"
        f"Question: {question}\n\n"
        "Final consolidated answer (cite exact fund names, share counts, CUSIPs, "
        "and filing dates for every claim):"
    )
    final = _call_gemini(reduce_system, reduce_user, max_tokens=1024)
    return final, map_log


def run_enhanced_rag(tq, docs):
    """Route to Map-Reduce for comparative/synthesis queries; Stuff chain for all others."""
    if tq.query_type in MAP_REDUCE_TYPES:
        final_answer, map_log = run_map_reduce_rag(tq.question, docs)
        return final_answer, "map_reduce", map_log

    final_answer = run_stuff_rag(tq.question, docs)
    return final_answer, "stuff", []



print("Architecture:")
print("Entity-scoped filter         : ACTIVE (post-retrieval, restricts to target fund(s)).")

print(f"  Generator  : {GENERATOR_MODEL_NAME} via Google AI Studio (Google DeepMind)")
print("Chunk pre-filtering          : ACTIVE (keyword row filter for large tables).")

print(f"  Evaluator  : {EVALUATOR_MODEL} via GitHub Models (OpenAI)")
print("Grounding check              : Non-vacuous (refusals → indeterminate, not True).")

print(f"  Judge      : {JUDGE_MODEL} via Google AI Studio (Google DeepMind)")
print(f"Judge inter-call delay       : {_JUDGE_INTER_CALL_DELAY}s.")

print("  → Two different providers (Google / OpenAI) — no self-grading bias.")
print(f"Evaluator inter-call delay   : {_EVALUATOR_INTER_CALL_DELAY}s.")

print("Generator default max_tokens : 1024.")
print(f"Gemini gen inter-call delay  : {_GEMINI_GEN_INTER_CALL_DELAY}s.")

C:\Users\user\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\user\AppData\Local\Temp\ipykernel_22868\1717398366.py:15: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Gemini API key loaded from: gemini_api_key.txt
GitHub token loaded from: github_token.txt
Architecture:
Entity-scoped filter         : ACTIVE (post-retrieval, restricts to target fund(s)).
  Generator  : gemini-3.1-flash-lite-preview via Google AI Studio (Google DeepMind)
Chunk pre-filtering          : ACTIVE (keyword row filter for large tables).
  Evaluator  : gpt-4o-mini via GitHub Models (OpenAI)
Grounding check              : Non-vacuous (refusals → indeterminate, not True).
  Judge      : gemini-2.5-pro via Google AI Studio (Google DeepMind)
Judge inter-call delay       : 2s.
  → Two different providers (Google / OpenAI) — no self-grading bias.
Evaluator inter-call delay   : 2s.
Generator default max_tokens : 1024.
Gemini gen inter-call delay  : 2s.


In [8]:

# ── Diagnostic: Berkshire Hathaway & Apple chunk coverage (Improvement 5) ─────
# Verifies that the BM25 corpus and vector store contain BH holdings data with
# Apple / AAPL / CUSIP 037833100, required for Q01 and Q02 to succeed.
#
# If both checks warn, Q01/Q02 will score 1/5 regardless of generator or chain
# choice. The root cause would be in Stage 2 parsing (CIK 0001067983) or Stage 3
# indexing, and those stages must be re-run before evaluating.
#
# Run this cell once before the evaluation loop to surface retrieval gaps early.

_APPLE_TERMS = ["037833100", "apple inc", "aapl"]
_BH_TERMS    = ["berkshire", "0001067983"]

print("=" * 70)
print("DIAGNOSTIC: Berkshire Hathaway & Apple chunk coverage")
print("=" * 70)

# 1. BM25 corpus (rebuilt from processed_holdings.json each session)
_bh_docs_bm25 = [d for d in _docs if any(t in d.page_content.lower() for t in _BH_TERMS)]
_apple_in_bh  = [d for d in _bh_docs_bm25 if any(t in d.page_content.lower() for t in _APPLE_TERMS)]

print(f"\nBM25 corpus (processed_holdings.json):")
print(f"  Total docs                                    : {len(_docs)}")
print(f"  Docs mentioning Berkshire / CIK 0001067983    : {len(_bh_docs_bm25)}")
print(f"  Berkshire docs mentioning Apple/AAPL/037833100 : {len(_apple_in_bh)}")

if _apple_in_bh:
    print(f"\n  [OK] Sample BH + Apple chunk (first 500 chars):")
    print(f"  {_apple_in_bh[0].page_content[:500]}")
else:
    print("\n  [WARN] No Berkshire + Apple chunks found in BM25 corpus.")
    print("  Q01 and Q02 cannot succeed. Re-run Stage 2 for CIK 0001067983,")
    print("  then re-run Stage 3 to re-index into Chroma.")

# 2. Vector store similarity search
print(f"\nVector store — similarity search: 'Berkshire Hathaway Apple largest position'")
_vs_results = vectorstore.similarity_search(
    "Berkshire Hathaway Apple largest position", k=5
)
_bh_vs    = [d for d in _vs_results if any(t in d.page_content.lower() for t in _BH_TERMS)]
_apple_vs = [d for d in _bh_vs      if any(t in d.page_content.lower() for t in _APPLE_TERMS)]

print(f"  Top-5 results containing BH terms     : {len(_bh_vs)}")
print(f"  Of those, also containing Apple terms  : {len(_apple_vs)}")

if _apple_vs:
    print(f"\n  [OK] Sample vector result (first 400 chars):")
    print(f"  {_apple_vs[0].page_content[:400]}")
else:
    print("  [WARN] Vector retrieval does not surface BH + Apple chunks for Q01/Q02.")

print(f"\n{'=' * 70}")
print("Diagnostic complete.")


DIAGNOSTIC: Berkshire Hathaway & Apple chunk coverage

BM25 corpus (processed_holdings.json):
  Total docs                                    : 95377
  Docs mentioning Berkshire / CIK 0001067983    : 498
  Berkshire docs mentioning Apple/AAPL/037833100 : 68

  [OK] Sample BH + Apple chunk (first 500 chars):
  ## Fund: Berkshire Hathaway Inc (Warren Buffett)
**Reported Name:** BERKSHIRE HATHAWAY INC  
**Filing Date:** 2026-02-17  
**Period of Report:** 2025-12-31  
**Accession Number:** 0001193125-26-054580  
**CIK:** 0001067983

| nameOfIssuer        | titleOfClass   | cusip     |       value |   sshPrnamt | investmentDiscretion   |
|:--------------------|:---------------|:----------|------------:|------------:|:-----------------------|
| ALLY FINL INC       | COM            | 02005N100 |   576074081

Vector store — similarity search: 'Berkshire Hathaway Apple largest position'
  Top-5 results containing BH terms     : 5
  Of those, also containing Apple terms  : 0
  [WARN] Vector retr

In [9]:

# ── Quick health check ────────────────────────────────────────────────────────
# Tests Gemini 3.1 Flash-Lite (generator), GPT-4o-mini evaluator,
# and Gemini 2.5 Pro (judge) with minimal token use.

import time as _time

def _probe_gemini_generator(attempt: int = 1):
    """Probe Gemini 3.1 Flash-Lite generator with a minimal request."""
    try:
        resp = _gemini_generator.generate_content("hi")
        print(f"  {GENERATOR_MODEL_NAME:40s}  OK (response received)")
        print(f"    Preview: {resp.text[:80]}...")
    except Exception as exc:
        err_str = str(exc)
        is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
        if is_rate_limit and attempt < 3:
            print(f"  {GENERATOR_MODEL_NAME:40s}  RATE LIMITED  [attempt {attempt}]")
            print(f"    -> Waiting 10s then retrying ...")
            _time.sleep(10)
            _probe_gemini_generator(attempt + 1)
        else:
            print(f"  {GENERATOR_MODEL_NAME:40s}  ERROR: {exc}")

def _probe_github_model(model_name, role_label):
    """Probe a GitHub Models endpoint with a minimal request."""
    try:
        resp = _github_client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=5,
        )
        reply = resp.choices[0].message.content.strip()
        print(f"  {model_name:40s}  OK (response: {reply[:40]})")
    except Exception as exc:
        print(f"  {model_name:40s}  ERROR: {exc}")

def _probe_gemini_judge(attempt: int = 1):
    """Probe Gemini 2.5 Pro judge with a minimal request."""
    try:
        resp = _gemini_judge.generate_content("hi")
        print(f"  {JUDGE_MODEL:40s}  OK (response received)")
        print(f"    Preview: {resp.text[:80]}...")
    except Exception as exc:
        err_str = str(exc)
        is_rate_limit = "429" in err_str or "RESOURCE_EXHAUSTED" in err_str
        if is_rate_limit and attempt < 3:
            print(f"  {JUDGE_MODEL:40s}  RATE LIMITED  [attempt {attempt}]")
            print(f"    -> Waiting 10s then retrying ...")
            _time.sleep(10)
            _probe_gemini_judge(attempt + 1)
        else:
            print(f"  {JUDGE_MODEL:40s}  ERROR: {exc}")

print("=" * 60)
print("Health Check")
print("=" * 60)
print(f"\n[Generator — Google AI Studio]")
_probe_gemini_generator()
print(f"\n[Evaluator — GitHub Models (OpenAI)]")
_time.sleep(1)
_probe_github_model(EVALUATOR_MODEL, "Evaluator")
print(f"\n[Judge — Google AI Studio (Gemini 2.5 Pro)]")
_time.sleep(1)
_probe_gemini_judge()
print("\n" + "=" * 60)

Health Check

[Generator — Google AI Studio]
  gemini-3.1-flash-lite-preview             OK (response received)
    Preview: Hello! How can I help you today?...

[Evaluator — GitHub Models (OpenAI)]
  gpt-4o-mini                               OK (response: Hello! How can I)

[Judge — Google AI Studio (Gemini 2.5 Pro)]
  gemini-2.5-pro                            OK (response received)
    Preview: Hello! How can I help you today?...



In [10]:

# ── Run evaluation loop ────────────────────────────────────────────────────────
# Generator  : Gemini 3.1 Flash-Lite via Google AI Studio (Google DeepMind)
# Evaluator  : GPT-4o-mini via GitHub Models (OpenAI)    — 150 req/day
# Judge      : Gemini 2.5 Pro via Google AI Studio (Google DeepMind)
# Routing    : Stuff chain for exact-lookup queries, Map-Reduce for synthesis queries
#
# Each answer is scored independently by BOTH evaluator and judge.
# The final faithfulness score is the average of both scores.

# Reset session counters at the start of each eval run
_gemini_gen_call_count = 0
_evaluator_call_count = 0
_judge_call_count = 0

results = []

print(f"Starting evaluation of {len(TEST_SET)} queries.")
print(f"Generator : {GENERATOR_MODEL_NAME} (Google AI Studio)")
print(f"Evaluator : {EVALUATOR_MODEL} (GitHub Models / OpenAI)")
print(f"Judge     : {JUDGE_MODEL} (Google AI Studio)")
print(f"This run will use ~{len(TEST_SET) * 2} calls per judge model "
      f"({len(TEST_SET)} queries × 2 answers each).")

for tq in TEST_SET:
    print(f"\n{'='*70}")
    print(f"Q{tq.id:02d} [{tq.query_type}]  "
          f"[Eval calls: {_evaluator_call_count} | Judge calls: {_judge_call_count}]")
    print(f"Question: {tq.question}")
    print(f"{'='*70}")

    # ── RETRIEVE context FIRST so both baseline and enhanced are judged ───────
    # against the SAME retrieved chunks.
    chain_label = "Map-Reduce" if tq.query_type in MAP_REDUCE_TYPES else "Stuff"

    # ── Select narrow or wide reranker based on query scope (Improvement 15) ──
    target_funds = _extract_fund_entities(tq.question)
    is_broad = len(target_funds) == 0
    _retriever = reranking_retriever_wide if (is_broad and tq.query_type in MAP_REDUCE_TYPES) \
                 else reranking_retriever
    print(f"  Retrieving context ({'wide' if _retriever is reranking_retriever_wide else 'narrow'} reranker) ...")
    try:
        rag_docs = _retriever.invoke(tq.question)
    except Exception as exc:
        print(f"  [WARN] Reranker failed ({exc}); falling back to hybrid retriever")
        rag_docs = hybrid_retriever.invoke(tq.question)[:8]
    print(f"  Retrieved {len(rag_docs)} chunks (pre-filter).")

    # ── ENTITY-SCOPED FILTER (Improvement 12) ────────────────────────────────
    if target_funds:
        _pre_count = len(rag_docs)
        rag_docs = _entity_filter(rag_docs, target_funds)
        print(f"  Entity filter: {sorted(target_funds)} → {len(rag_docs)}/{_pre_count} chunks retained")
    else:
        print(f"  Entity filter: broad query — all {len(rag_docs)} chunks retained")

    # ── TEMPORAL FILTER (Improvement 13) ──────────────────────────────────────
    _pre_temporal = len(rag_docs)
    rag_docs = _temporal_filter(rag_docs, tq.question)
    if len(rag_docs) != _pre_temporal:
        print(f"  Temporal filter: {len(rag_docs)}/{_pre_temporal} chunks retained")

    # ── FUND DIVERSITY FILTER (Improvement 14) ────────────────────────────────
    if is_broad:
        _pre_diverse = len(rag_docs)
        rag_docs = _diversify_by_fund(rag_docs, max_per_fund=2)
        if len(rag_docs) != _pre_diverse:
            print(f"  Diversity filter: {len(rag_docs)}/{_pre_diverse} chunks retained")

    # ── BASELINE: Gemini 3.1 Flash-Lite with NO retrieved context ─────────────
    print(f"  Running BASELINE ({GENERATOR_MODEL_NAME}, no context) ...")
    try:
        baseline_resp = run_baseline(tq.question)
    except Exception as exc:
        baseline_resp = f"[LLM ERROR] {exc}"

    # Dual-judge baseline against the SAME retrieved context
    b_avg, b_eval_s, b_eval_r, b_eval_m, b_judge_s, b_judge_r, b_judge_m = \
        score_dual(baseline_resp, tq, rag_docs)
    b_grounded, b_grounding_detail = grounding_check(baseline_resp, FILINGS_BASE)
    print(f"  Baseline: Eval={b_eval_s}/5 ({b_eval_m})  Judge={b_judge_s}/5 ({b_judge_m})  "
          f"Avg={b_avg}  Grounded={b_grounded}")

    # ── ENHANCED: Hybrid BM25+Vector + Reranker + Gemini (routed chain) ───────
    print(f"  Running ENHANCED RAG "
          f"({GENERATOR_MODEL_NAME} / {chain_label}, "
          f"{'Hybrid+Reranker' if USE_RERANKER else 'Hybrid'}) ...")

    try:
        rag_resp, _chain_method, map_log = run_enhanced_rag(tq, rag_docs)
    except Exception as exc:
        rag_resp = f"[CHAIN ERROR] {exc}"
        map_log = []

    # Dual-judge enhanced
    r_avg, r_eval_s, r_eval_r, r_eval_m, r_judge_s, r_judge_r, r_judge_m = \
        score_dual(rag_resp, tq, rag_docs)
    r_grounded, r_grounding_detail = grounding_check(rag_resp, FILINGS_BASE)
    _grounded_label = {True: "YES", False: "NO", None: "N/A (refusal)"}[r_grounded]
    print(f"  Enhanced: Eval={r_eval_s}/5 ({r_eval_m})  Judge={r_judge_s}/5 ({r_judge_m})  "
          f"Avg={r_avg}  Grounded={_grounded_label}")

    # ── Detect failure modes ──────────────────────────────────────────────────
    failure_modes = []
    if r_avg <= 2:
        failure_modes.append("Low faithfulness — response lacks specific citations.")
    if tq.expect_no_data and r_avg < 4:
        failure_modes.append("Hallucination on edge-case (non-existent entity/period).")
    if r_grounded is False:
        failure_modes.append(f"Grounding failure: {r_grounding_detail}")
    if r_grounded is None and not tq.expect_no_data:
        failure_modes.append(f"Refusal despite available context: {r_grounding_detail}")
    if "not available" in rag_resp.lower() and not tq.expect_no_data:
        failure_modes.append("Enhanced system failed to retrieve context for a valid query.")

    results.append({
        "id"                 : tq.id,
        "type"               : tq.query_type,
        "question_short"     : (tq.question[:57] + "...") if len(tq.question) > 57 else tq.question,
        # Averaged scores (primary)
        "baseline_score"     : b_avg,
        "enhanced_score"     : r_avg,
        "delta"              : round(r_avg - b_avg, 1),
        "enhanced_grounded"  : r_grounded,
        # Individual evaluator scores (GPT-4o-mini)
        "b_eval_score"       : b_eval_s,
        "b_eval_rationale"   : b_eval_r,
        "b_eval_method"      : b_eval_m,
        "r_eval_score"       : r_eval_s,
        "r_eval_rationale"   : r_eval_r,
        "r_eval_method"      : r_eval_m,
        # Individual judge scores (Gemini 2.5 Pro)
        "b_judge_score"      : b_judge_s,
        "b_judge_rationale"  : b_judge_r,
        "b_judge_method"     : b_judge_m,
        "r_judge_score"      : r_judge_s,
        "r_judge_rationale"  : r_judge_r,
        "r_judge_method"     : r_judge_m,
        # Responses and metadata
        "baseline_response"  : baseline_resp,
        "enhanced_response"  : rag_resp,
        "b_grounding_detail" : b_grounding_detail,
        "r_grounding_detail" : r_grounding_detail,
        "failure_modes"      : failure_modes,
        "chain_type"         : chain_label,
        "map_summaries"      : map_log,
    })

print(f"\n\nEvaluation complete.  {len(results)} queries processed.")
print(f"Evaluator calls ({EVALUATOR_MODEL}): {_evaluator_call_count}")
print(f"Judge calls ({JUDGE_MODEL}): {_judge_call_count}")

_total_judge_calls = _evaluator_call_count + _judge_call_count
_expected = len(TEST_SET) * 4  # 2 answers × 2 judges per query
_fallbacks = _expected - _total_judge_calls
if _fallbacks > 0:
    print(f"WARNING: {_fallbacks} of {_expected} evaluations used keyword fallback (rate limit hit).")
    print("         Scores may be unreliable. Wait for quota reset before interpreting results.")
else:
    print("All evaluations used the dual-judge system — scores are reliable.")

# ── Persist evaluation results to disk ────────────────────────────────────────
from datetime import datetime, timezone as _tz
_EVAL_OUTPUT_PATH = "./data/evaluation_results.json"
_serialisable = []
for _r in results:
    _sr = dict(_r)
    if _sr.get("enhanced_grounded") is None:
        _sr["enhanced_grounded"] = "indeterminate"
    _serialisable.append(_sr)
_eval_payload = {
    "timestamp": datetime.now(_tz.utc).isoformat(),
    "generator": GENERATOR_MODEL_NAME,
    "evaluator": EVALUATOR_MODEL,
    "judge": JUDGE_MODEL,
    "evaluator_calls": _evaluator_call_count,
    "judge_calls": _judge_call_count,
    "results": _serialisable,
}
with open(_EVAL_OUTPUT_PATH, "w", encoding="utf-8") as _ef:
    json.dump(_eval_payload, _ef, indent=2, ensure_ascii=False)
print(f"\nResults persisted to: {_EVAL_OUTPUT_PATH} ({os.path.getsize(_EVAL_OUTPUT_PATH):,} bytes)")


Starting evaluation of 10 queries.
Generator : gemini-3.1-flash-lite-preview (Google AI Studio)
Evaluator : gpt-4o-mini (GitHub Models / OpenAI)
Judge     : gemini-2.5-pro (Google AI Studio)
This run will use ~20 calls per judge model (10 queries × 2 answers each).

Q01 [Fact-based]  [Eval calls: 0 | Judge calls: 0]
Question: What is Berkshire Hathaway's largest position by market value in its Feb 2026 filing?
  Retrieving context (narrow reranker) ...
  Retrieved 8 chunks (pre-filter).
  Entity filter: ['Berkshire Hathaway Inc (Warren Buffett)'] → 2/8 chunks retained
  Temporal filter: 1/2 chunks retained
  Running BASELINE (gemini-3.1-flash-lite-preview, no context) ...
  Baseline: Eval=2/5 (gpt4omini_evaluator)  Judge=1/5 (gemini2_judge)  Avg=1.5  Grounded=True
  Running ENHANCED RAG (gemini-3.1-flash-lite-preview / Stuff, Hybrid+Reranker) ...
  Enhanced: Eval=5/5 (gpt4omini_evaluator)  Judge=5/5 (gemini2_judge)  Avg=5.0  Grounded=YES

Q02 [Numerical Extraction]  [Eval calls: 2 | Ju

INFO:openai._base_client:Retrying request to /chat/completions in 60.000000 seconds


  Enhanced: Eval=4/5 (gpt4omini_evaluator)  Judge=2/5 (gemini2_judge)  Avg=3.0  Grounded=YES

Q10 [Temporal — Specific Fund Period]  [Eval calls: 18 | Judge calls: 18]
Question: What filing date appears on the Scion Asset Management (Michael Burry) 13F submission, and does it confirm activity in the February 2026 filing window?
  Retrieving context (narrow reranker) ...
  Retrieved 8 chunks (pre-filter).
  Entity filter: ['Scion Asset Management (Michael Burry)'] → 4/8 chunks retained
  Running BASELINE (gemini-3.1-flash-lite-preview, no context) ...
  Baseline: Eval=2/5 (gpt4omini_evaluator)  Judge=1/5 (gemini2_judge)  Avg=1.5  Grounded=True
  Running ENHANCED RAG (gemini-3.1-flash-lite-preview / Stuff, Hybrid+Reranker) ...
  Enhanced: Eval=4/5 (gpt4omini_evaluator)  Judge=5/5 (gemini2_judge)  Avg=4.5  Grounded=N/A (refusal)


Evaluation complete.  10 queries processed.
Evaluator calls (gpt-4o-mini): 20
Judge calls (gemini-2.5-pro): 20
All evaluations used the dual-judge system — scor

In [11]:
# ── Print full responses for each query ───────────────────────────────────────
WRAP = 100

for r in results:
    print(f"\n{'='*70}")
    print(f"Q{r['id']:02d} | {r['type']}")
    print(f"Question: {r['question_short']}")
    print(f"{'─'*70}")

    print("BASELINE RESPONSE:")
    print(textwrap.fill(r['baseline_response'].strip(), width=WRAP))
    print(f"  Avg Score: {r['baseline_score']}/5")
    print(f"    Evaluator ({EVALUATOR_MODEL}): {r['b_eval_score']}/5 — {r['b_eval_rationale']}")
    print(f"    Judge ({JUDGE_MODEL}): {r['b_judge_score']}/5 — {r['b_judge_rationale']}")
    print(f"  Grounding: {r['b_grounding_detail']}")

    print(f"{'─'*70}")
    print(f"ENHANCED (Hybrid RAG + Reranker + {GENERATOR_MODEL_NAME} / {r.get('chain_type','Stuff')}) RESPONSE:")
    print(textwrap.fill(r['enhanced_response'].strip(), width=WRAP))
    print(f"  Avg Score: {r['enhanced_score']}/5")
    print(f"    Evaluator ({EVALUATOR_MODEL}): {r['r_eval_score']}/5 — {r['r_eval_rationale']}")
    print(f"    Judge ({JUDGE_MODEL}): {r['r_judge_score']}/5 — {r['r_judge_rationale']}")
    print(f"  Grounding: {r['r_grounding_detail']}")

    if r['failure_modes']:
        print(f"  FAILURE MODES:")
        for fm in r['failure_modes']:
            print(f"      - {fm}")



Q01 | Fact-based
Question: What is Berkshire Hathaway's largest position by market v...
──────────────────────────────────────────────────────────────────────
BASELINE RESPONSE:
As a financial analyst, I must clarify that it is currently impossible to provide the data for a
**February 2026** 13F-HR filing.  13F-HR filings are submitted to the SEC by institutional
investment managers on a quarterly basis, typically within 45 days after the end of each calendar
quarter. Therefore, the filing for the period ending December 31, 2025, will not be submitted until
**mid-February 2026**.  Because we are currently in 2024, the data for the February 2026 filing does
not yet exist.
  Avg Score: 1.5/5
    Evaluator (gpt-4o-mini): 2/5 — The answer incorrectly states that the February 2026 filing data does not exist, despite the context providing specific details about Berkshire Hathaway's holdings as of December 31, 2025.
    Judge (gemini-2.5-pro): 1/5 — The answer incorrectly refuses to answer t

In [12]:
# ── Summary comparison table ───────────────────────────────────────────────────
table_rows = [
    [
        f"Q{r['id']:02d}",
        r['type'][:28],
        r['baseline_score'],
        f"{r['b_eval_score']}/{r['b_judge_score']}",
        r['enhanced_score'],
        f"{r['r_eval_score']}/{r['r_judge_score']}",
        f"+{r['delta']}" if r['delta'] > 0 else str(r['delta']),
        {True: "YES", False: "NO", None: "N/A"}[r['enhanced_grounded']],
        ("; ".join(r['failure_modes'])[:35] + "...") if len("; ".join(r['failure_modes'])) > 35
        else ("; ".join(r['failure_modes']) if r['failure_modes'] else "None"),
    ]
    for r in results
]

avg_baseline = sum(r['baseline_score'] for r in results) / len(results)
avg_enhanced = sum(r['enhanced_score'] for r in results) / len(results)
avg_delta    = avg_enhanced - avg_baseline

table_rows.append([
    "AVG", "",
    f"{avg_baseline:.1f}", "",
    f"{avg_enhanced:.1f}", "",
    f"+{avg_delta:.1f}" if avg_delta >= 0 else f"{avg_delta:.1f}",
    "", ""
])

headers = [
    "Query", "Type", "B-Avg", "B(E/J)", "E-Avg", "E(E/J)", "Delta",
    "Ground?", "Failure Modes"
]

# Count keyword fallbacks
_eval_fb = sum(1 for r in results if r.get("b_eval_method") == "keyword_fallback") + \
           sum(1 for r in results if r.get("r_eval_method") == "keyword_fallback")
_judge_fb = sum(1 for r in results if r.get("b_judge_method") == "keyword_fallback") + \
            sum(1 for r in results if r.get("r_judge_method") == "keyword_fallback")
_total_fb = _eval_fb + _judge_fb
_total_evals = len(results) * 4  # 2 answers × 2 judges

print("\n" + "═" * 100)
print("EVALUATION SUMMARY: Baseline vs. Enhanced Hybrid RAG (Dual-Judge)")
print("═" * 100)

if _total_fb > 0:
    print()
    print("  ⚠  RELIABILITY WARNING  ⚠")
    print(f"  {_total_fb} of {_total_evals} evaluations fell back to the keyword scorer")
    print(f"  due to rate limits on GitHub Models.")
    print(f"  Keyword fallback scores should be treated as UNRELIABLE.")
    print("  ► Re-run this cell after the quota resets.")
    print()

print(tabulate(table_rows, headers=headers, tablefmt="rounded_outline"))
print()
print(f"B(E/J) = Baseline scores: Evaluator / Judge")
print(f"E(E/J) = Enhanced scores: Evaluator / Judge")
print()
print(f"Baseline avg faithfulness  : {avg_baseline:.1f} / 5")
print(f"Enhanced avg faithfulness  : {avg_enhanced:.1f} / 5")
print(f"Overall improvement        : {avg_delta:+.1f} points")
print(f"Generator                  : {GENERATOR_MODEL_NAME} via Google AI Studio (Google DeepMind)")
print(f"Evaluator (faithfulness)   : {EVALUATOR_MODEL} via GitHub Models (OpenAI)")
print(f"Judge (faithfulness)       : {JUDGE_MODEL} via Google AI Studio (Google DeepMind)")
print(f"  → Two providers (Google / OpenAI) — no self-grading bias.")
print(f"Evaluator fallbacks        : {_eval_fb} / {_total_evals // 2}")
print(f"Judge fallbacks            : {_judge_fb} / {_total_evals // 2}")



════════════════════════════════════════════════════════════════════════════════════════════════════
EVALUATION SUMMARY: Baseline vs. Enhanced Hybrid RAG (Dual-Judge)
════════════════════════════════════════════════════════════════════════════════════════════════════
╭─────────┬──────────────────────────────┬─────────┬──────────┬─────────┬──────────┬─────────┬───────────┬────────────────────────────────────────╮
│ Query   │ Type                         │   B-Avg │ B(E/J)   │   E-Avg │ E(E/J)   │   Delta │ Ground?   │ Failure Modes                          │
├─────────┼──────────────────────────────┼─────────┼──────────┼─────────┼──────────┼─────────┼───────────┼────────────────────────────────────────┤
│ Q01     │ Fact-based                   │     1.5 │ 2/1      │     5   │ 5/5      │     3.5 │ YES       │ None                                   │
│ Q02     │ Numerical Extraction         │     1   │ 1/1      │     4.5 │ 4/5      │     3.5 │ YES       │ None                            